# Cold-start short-video recommendation: end to end

Runs the whole pipeline on a free Colab T4: download, tests, embeddings, all models x 3 seeds, LoRA, report.

**Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
REPO_URL = "https://github.com/FireCloud24/shortvideo-coldstart.git"  # change if your fork lives elsewhere
SEEDS = "0 1 2"

import os
if not os.path.exists('shortvideo-coldstart'):
    !git clone -q $REPO_URL
%cd shortvideo-coldstart
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 1. Tests

In [ ]:
!python -m pytest -q

## 2. Download MicroLens-100K (T2)
About 710 MB. Re-running skips files already present.

In [ ]:
!python scripts/download.py
!cat data/raw/manifest.json | head -20

## 3. Embeddings (T6)
A 64-item check first, so a problem shows up in a minute rather than an hour.

In [ ]:
!python scripts/embed.py --model cnclip --limit 64
!python scripts/embed.py --model qwen --limit 32

In [ ]:
!python scripts/embed.py --model cnclip
!python scripts/embed.py --model qwen
!cat data/features/*.meta.json

## 4. Baselines and content models (T5, T7)

In [ ]:
CONFIGS = ['popularity', 'sasrec_id', 'content_cnclip_image', 'content_cnclip_text',
           'content_cnclip_image_text', 'content_qwen_vl',
           'id_content_cnclip_image_text', 'id_content_qwen_vl']
for name in CONFIGS:
    seeds = '0' if name == 'popularity' else SEEDS  # popularity has no randomness
    !python scripts/train.py --config configs/{name}.yaml --seeds {seeds}

## 5. LoRA fine-tuning (T8)
Text tower by default. If it runs out of memory, lower `--batch-size`, or use the adapter fallback in the next cell instead.

In [ ]:
!python scripts/finetune.py --mode lora --modality text
!python scripts/embed.py --model cnclip --what text --lora-dir checkpoints/lora_text --tag lora
!python scripts/train.py --config configs/content_cnclip_text_lora.yaml --seeds {SEEDS}

In [ ]:
# Fallback / extra comparison: 2-layer adapter on the cached image features.
!python scripts/finetune.py --mode adapter --feature cnclip_image
!python scripts/train.py --config configs/content_cnclip_image.yaml --seeds {SEEDS} --set name=content_cnclip_image_adapter "model.features=[cnclip_image_adapter]"

## 6. Report (T9)

In [ ]:
!python scripts/report.py
from IPython.display import Markdown, Image, display
display(Markdown(open('results/results.md').read()))
display(Image('results/warm_vs_cold.png'))
display(Markdown(open('results/error_slices.md').read()))
display(Image('results/error_slices.png'))

## 7. Save the results
`results/` holds only metrics, tables and charts (no dataset files), so it can be committed. Download the zip and commit it from your machine.

In [ ]:
!zip -q -r results.zip results
from google.colab import files
files.download('results.zip')